# dWGS — Part 8: Data Requirements (NEY GMS → NW GMS, RGL to SGL)

NHS England is transitioning centralised Whole Genome Sequencing (WGS) to a distributed
model (dWGS): each NHS GMS geography's **Requesting Genomic Laboratory (RGL)** submits
DNA samples directly to a **Sequencing Genomic Laboratory (SGL)**. This notebook covers
a specific subcontracted arrangement: **North East and Yorkshire GMS acting as RGL**,
submitting samples and a digital manifest to **NW GMS acting as SGL**, whose inbound
order pathway is the same NW Regional Integration Engine (RIE) → iGene ORM^O01
interface `03`-`07` already work with.

Three source documents define the requirement (all in `NotGit/`, gitignored per this
repo's convention, referenced here by name only):

- **`RGL to SGL SOP v0.4.docx`** — NHS England's SOP; Appendix 3 defines the 37-column
  national Digital Manifest CSV.
- **`dWGS Manifest CSV to HL7v2 ORM Transformation Specification v0.1.docx`** — NW GMS's
  draft spec turning that manifest into an ORM^O01 message, plus a 5-column local
  extension NEY GMS adds for NW GMS's benefit.
- **`dWGS Sample Manifest HL7v2 Data Mapping v0.1.xlsx`** — the field-by-field working
  sheet those tables were drawn from.

This is a preparation notebook: the sample data (`Input/dWGS.csv`) and the three-way
field mapping below.

## Where this sits: from clinical order to sequencing result

This notebook's `LAB-35` sub-order is one hop in a longer chain that starts and ends
with the referring clinician. Mapped onto [NW-GMSA's Inter-Laboratory Workflow
(ILW)](https://nw-gmsa.github.io/en/ILW.html#sub-orders-lab-35-and-lab-36) actors:

| Role (ILW) | Party here | Transaction it sends |
|---|---|---|
| Order Placer | Test Ordering Entity (referring Trust/clinician) | `LAB-1` |
| Order Filler | Requesting Genomic Laboratory (RGL) | `LAB-35` (as sub-order placer), `LAB-3` |
| Sub Contractor | Sequencing Genomic Laboratory (SGL) | `LAB-36` *(Option B, below)* |
| Automation Manager | Genomics England Limited (GEL) | `LAB-5` *(Option A, below)* |

`LAB-1` (the original clinical order) and `LAB-35` (RGL's sub-order to the SGL) are
settled — that's what this notebook builds. What's still open is how the result of
that sub-order gets back to the RGL, because GEL sits behind the SGL in the physical
pathway (RGL to SGL SOP v0.4, Figure 1) and doesn't map cleanly onto a single ILW role.

### Process flow

![notebook 8 diagram 1](https://mermaid.ink/svg/Zmxvd2NoYXJ0IFRECiAgICBPUFsiT3JkZXIgUGxhY2VyXG5UZXN0IE9yZGVyaW5nIEVudGl0eSJdCiAgICBPRlsiT3JkZXIgRmlsbGVyXG5SZXF1ZXN0aW5nIEdlbm9taWMgTGFib3JhdG9yeSAoUkdMKSJdCgogICAgc3ViZ3JhcGggU0VRWyJTZXF1ZW5jaW5nIl0KICAgICAgICBTQ1siU3ViIENvbnRyYWN0b3JcblNlcXVlbmNpbmcgR2Vub21pYyBMYWJvcmF0b3J5IChTR0wpIl0KICAgICAgICBHRUxbIkdlbm9taWNzIEVuZ2xhbmQgTGltaXRlZFxuKEF1dG9tYXRpb24gTWFuYWdlcikiXQogICAgICAgIFNDIC0tICJwcmVwYXJlZCBzYW1wbGUiIC0tPiBHRUwKICAgIGVuZAoKICAgIE9QIC0tICJMQUItMVxubGFib3JhdG9yeSBvcmRlciIgLS0+IE9GCiAgICBPRiAtLSAiTEFCLTM1XG5zdWItb3JkZXIgKyBtYW5pZmVzdCIgLS0+IFNDCiAgICBHRUwgLS4gIkxBQi01IChPcHRpb24gQSlcbmRpcmVjdCB0byBPcmRlciBGaWxsZXIiIC4tPiBPRgogICAgU0MgLS4gIkxBQi0zNiAoT3B0aW9uIEIpXG52aWEgU3ViIENvbnRyYWN0b3IiIC4tPiBPRgogICAgT0YgLS0gIkxBQi0zXG5sYWJvcmF0b3J5IHJlcG9ydCIgLS0+IE9Q)

<!--
```mermaid
flowchart TD
    OP["Order Placer\nTest Ordering Entity"]
    OF["Order Filler\nRequesting Genomic Laboratory (RGL)"]

    subgraph SEQ["Sequencing"]
        SC["Sub Contractor\nSequencing Genomic Laboratory (SGL)"]
        GEL["Genomics England Limited\n(Automation Manager)"]
        SC -- "prepared sample" --> GEL
    end

    OP -- "LAB-1\nlaboratory order" --> OF
    OF -- "LAB-35\nsub-order + manifest" --> SC
    GEL -. "LAB-5 (Option A)\ndirect to Order Filler" .-> OF
    SC -. "LAB-36 (Option B)\nvia Sub Contractor" .-> OF
    OF -- "LAB-3\nlaboratory report" --> OP
```
-->

### Sequence diagram

![notebook 8 diagram 2](https://mermaid.ink/svg/c2VxdWVuY2VEaWFncmFtCiAgICBhY3RvciBPUCBhcyBPcmRlciBQbGFjZXI8YnIvPihUZXN0IE9yZGVyaW5nIEVudGl0eSkKICAgIHBhcnRpY2lwYW50IE9GIGFzIE9yZGVyIEZpbGxlcjxici8+KFJHTCkKICAgIHBhcnRpY2lwYW50IFNDIGFzIFN1YiBDb250cmFjdG9yPGJyLz4oU0dMKQogICAgcGFydGljaXBhbnQgR0VMIGFzIEdlbm9taWNzIEVuZ2xhbmQgTHRkPGJyLz4oQXV0b21hdGlvbiBNYW5hZ2VyKQoKICAgIE9QLT4+T0Y6IExBQi0xIExhYm9yYXRvcnkgT3JkZXIKICAgIE9GLT4+U0M6IExBQi0zNSBTdWItb3JkZXIgKHNhbXBsZSArIGRpZ2l0YWwgbWFuaWZlc3QpCiAgICBTQy0+PkdFTDogUmF3IGdlbm9taWMgZGF0YSAocG9zdC1zZXF1ZW5jaW5nKQoKICAgIGFsdCBPcHRpb24gQSDigJQgR0VMIGFzIEF1dG9tYXRpb24gTWFuYWdlcgogICAgICAgIEdFTC0+Pk9GOiBMQUItNSBzdHJ1Y3R1cmVkIHJlc3VsdCAoZGlyZWN0IHRvIE9yZGVyIEZpbGxlcikKICAgIGVsc2UgT3B0aW9uIEIg4oCUIHJlc3VsdCByZXR1cm5lZCB2aWEgdGhlIFN1YiBDb250cmFjdG9yCiAgICAgICAgR0VMLT4+U0M6IEFuYWx5c2VkL2ludGVncmF0ZWQgcmVzdWx0CiAgICAgICAgU0MtPj5PRjogTEFCLTM2IFN1Yi1vcmRlciByZXN1bHQKICAgIGVuZAoKICAgIE9GLT4+T1A6IExBQi0zIExhYm9yYXRvcnkgcmVwb3J0)

<!--
```mermaid
sequenceDiagram
    actor OP as Order Placer<br/>(Test Ordering Entity)
    participant OF as Order Filler<br/>(RGL)
    participant SC as Sub Contractor<br/>(SGL)
    participant GEL as Genomics England Ltd<br/>(Automation Manager)

    OP->>OF: LAB-1 Laboratory Order
    OF->>SC: LAB-35 Sub-order (sample + digital manifest)
    SC->>GEL: Raw genomic data (post-sequencing)

    alt Option A — GEL as Automation Manager
        GEL->>OF: LAB-5 structured result (direct to Order Filler)
    else Option B — result returned via the Sub Contractor
        GEL->>SC: Analysed/integrated result
        SC->>OF: LAB-36 Sub-order result
    end

    OF->>OP: LAB-3 Laboratory report
```
-->

The rest of this notebook zooms into the `LAB-35` step: NE&Y Genomics acting as RGL
(the *Order Filler* above, confusingly also acting as *sub-order placer* toward NW
Genomics), and NW Genomics acting as SGL (*Sub Contractor* above). The field mapping,
worked example, and message-exchange sections below are all about that one hop, not
the full pathway shown here.

## Field mapping: CSV → HL7v2 → FHIR

All 42 manifest fields (37 national, from `RGL to SGL SOP v0.4` Appendix 3, plus 5 NEY
local-extension fields from the transformation spec Section 7.2). Fields the
transformation spec marks `HL7v2 Mapping Required = FALSE` are left blank in both the
`hl7v2_field` and `fhir_field` columns — they're carried in the CSV but not transmitted
onward.

The `fhir_field` column is this notebook's own addition — neither source document
mentions FHIR. Identifiers/profiles reused from `03`-`07` are shown as-is;
`family_structure` and `participant_type` have no home in the current NW-GMSA IG and are
marked `(proposed)`, and the NGIS identifier system is marked `(TBC)` since no published
FHIR system for it was found in either source document.

**Two specimens, not one**: the `primary_sample_*` fields describe the specimen as
originally received at the GLH (blood/tissue, before extraction); `dispatched_sample_*`
describes the extracted DNA sent onward. Each is its own `SPM` segment/`Specimen`
resource - the dispatched specimen's `Specimen.parent` would reference the primary
specimen's. `primary_sample_type`'s germline/tumour distinction has no clean SPM/Specimen
field to sit in (HL7v2 has no field for it, and FHIR's closer fit is arguably an
`Observation` component, not `Specimen`) - marked low-confidence rather than guessed.

**GLH codes are not trust codes**: `ordering_entity_id` is correctly a referring NHS
Trust's ODS code (e.g. `RR8` Leeds Teaching Hospitals) — but `glh_laboratory_id` is a
*Genomic Laboratory Hub* code, a separate ODS tier above individual trusts (the same
pattern `02-work-orders-worked-example.ipynb` explains for `699X0`/`K1S6S`, North West's
own hub/site codes). Confirmed directly against the live NHS ODS API
(`directory.spineservices.nhs.uk`): North East and Yorkshire's hub is
**`699N0`, "YORKSHIRE AND NORTH EAST GLH"** (Newcastle-based) — one GLH code shared
across every referring trust in that geography, not a copy of whichever trust happens to
be ordering.

In [1]:
import pandas as pd

FIELD_MAPPING = [
    # csv_field, common_name, cardinality, field_type, hl7v2_field, fhir_field
    # cardinality/field_type source: RGL to SGL SOP v0.4 Appendix 3 (national fields) and the
    # transformation spec Section 7.2 (NEY local-extension fields) - in Input/dWGS.csv column order.
    ("referral_id", "Original Order Placer Group Number", "MUST", "String",
     "OBX-5 (OBX-3=NGIS_REFERRAL_ID)", "ServiceRequest.requisition"),
    ("clinical_indication_test_type_id", "Test Code", "OPTIONAL", "String",
     "OBR-4.1", "ServiceRequest.code.coding (England-GenomicTestDirectory)"),
    ("patient_nhs_number", "NHS Number", "OPTIONAL", "String",
     "PID-3 (NH)", "Patient.identifier (NHS number)"),
    ("patient_ngis_id", "Patient Identifier", "MUST", "String",
     "PID-3 (NGIS)", "Patient.identifier.assigner (Genomics England, ODS 8J834)"),
    ("patient_date_of_birth", "Date Of Birth", "OPTIONAL", "Date",
     "PID-7.1", "Patient.birthDate"),
    ("ordering_entity_id", "Original Ordering Facility Code", "OPTIONAL", "Code (ODS Code)",
     "ORC-21", "Specimen.identifier (as received) assigner"),
    ("glh_laboratory_id", "Filler Order Ordering Facility Code", "MUST", "Code (ODS Code)",
     "ORC-21", "ServiceRequest.requester / requisition assigner / Specimen.identifier (LIMS) assigner"),
    ("primary_sample_received_date", "Sample Received Date", "OPTIONAL", "Date",
     "SPM-18 (primary SPM)", "Specimen.receivedTime (primary specimen)"),
    ("primary_sample_id_as_received_by_glh", "Received Sample Identifier", "OPTIONAL", "String",
     "SPM-2.1 (primary SPM)", "Specimen.identifier (as received)"),
    ("primary_sample_id_in_glh_lims", "LIMS Sample Identifier", "OPTIONAL", "String",
     "SPM-2.2 (primary SPM)", "Specimen.identifier (GLH LIMS)"),
    ("primary_sample_type", "Sample Type", "MUST", "Code (Specimen Type SNOMED CT)",
     "SPM-11 (primary SPM, low confidence)", "Specimen.type / extension (germline vs tumour, low confidence)"),
    ("primary_sample_state", "Sample Material Type", "MUST", "String (enum)",
     "SPM-4.1 (primary SPM)", "Specimen.type (primary specimen material)"),
    ("received_sample_topography", "Sample Topography", "MUST (cancer only)", "String", "", ""),
    ("received_sample_morphology", "Sample Morphology", "OPTIONAL", "String", "", ""),
    ("received_sample_tumour_content_%", "Tumour Content", "MUST (cancer only)", "Number", "", ""),
    ("received_sample_comments", "Sample Comments", "OPTIONAL", "String", "", ""),
    ("received_sample_collection_date", "Specimen Collection Date", "OPTIONAL", "Date", "", ""),
    ("dispatched_sample_id_in_glh_lims", "Dispatched Sample Identifier", "OPTIONAL", "String", "", ""),
    ("dispatched_sample_lsid", "Specimen Barcode", "MUST", "String",
     "SPM-2.1 and OBX-5 (OBX-3=DISPATCHED_SAMPLE_LSID)", "Specimen.container.identifier"),
    ("dispatched_sample_type", "Dispatched Sample Type", "MUST", "Code (Specimen Type SNOMED CT)", "", ""),
    ("dispatched_sample_state", "Dispatched Material Type", "MUST", "String (enum)",
     "SPM-4.1 (dispatched SPM)", "Specimen.type (text-only - DNA)"),
    ("dispatched_sample_volume_(ul)", "Sample Volume", "OPTIONAL", "Number", "", ""),
    ("laboratory_remaining_volume_banked_(ul)", "Remaining Banked Volume", "OPTIONAL", "Number", "", ""),
    ("glh_concentration_(ng/ul)", "DNA Concentration", "OPTIONAL", "Number", "", ""),
    ("glh_od260/280", "DNA Purity", "OPTIONAL", "Number", "", ""),
    ("glh_din_value", "DNA Integrity Number", "OPTIONAL", "Number", "", ""),
    ("glh_percentage_DNA_over_23kb", "DNA Fragment Size", "OPTIONAL", "Number", "", ""),
    ("glh_qc_status", "QC Status", "OPTIONAL", "String", "", ""),
    ("glh_sample_dispatch_date", "Dispatch Date", "OPTIONAL", "Date", "", ""),
    ("glh_sample_consignment_number", "Consignment Number", "OPTIONAL", "String", "", ""),
    ("plating_organisation", "Plating Organisation", "OPTIONAL", "Enum", "", ""),
    ("gmc_rack_id", "Rack Identifier", "OPTIONAL", "String", "", ""),
    ("gmc_rack_well", "Rack Well Position", "OPTIONAL", "String (pattern)", "", ""),
    ("dna_extraction_protocol", "DNA Extraction Method", "OPTIONAL", "String", "", ""),
    ("prolonged_sample_storage", "Sample Storage Method", "OPTIONAL", "String", "", ""),
    ("retrospective_sample", "Retrospective Sample Flag", "OPTIONAL", "Enum", "", ""),
    ("approved_by", "Approved By", "OPTIONAL", "String", "", ""),
    ("patient_forename", "Forename", "MUST", "String", "PID-5.2", "Patient.name.given"),
    ("patient_surname", "Surname", "MUST", "String", "PID-5.1", "Patient.name.family"),
    ("family_structure", "Family Structure", "MUST", "Enumerated string",
     "OBX-5 (OBX-3=FAMILY_STRUCTURE)", "Observation (via ServiceRequest.supportingInfo)"),
    ("participant_type", "Participant Type", "MUST", "Enumerated string",
     "OBX-5 (OBX-3=PARTICIPANT_TYPE)", "Observation (via ServiceRequest.supportingInfo)"),
    ("clinical_information", "Clinical Information", "OPTIONAL", "String", "NTE-3", "ServiceRequest.note"),
]

field_mapping_df = pd.DataFrame(
    FIELD_MAPPING,
    columns=["csv_field", "common_name", "cardinality", "field_type", "hl7v2_field", "fhir_field"],
)
field_mapping_df

,csv_field,common_name,cardinality,field_type,hl7v2_field,fhir_field
0,referral_id,Original Order Placer Group Number,MUST,String,OBX-5 (OBX-3=NGIS_REFERRAL_ID),ServiceRequest.requisition
1,clinical_indication_test_type_id,Test Code,OPTIONAL,String,OBR-4.1,ServiceRequest.code.coding (England-GenomicTes...
2,patient_nhs_number,NHS Number,OPTIONAL,String,PID-3 (NH),Patient.identifier (NHS number)
3,patient_ngis_id,Patient Identifier,MUST,String,PID-3 (NGIS),"Patient.identifier.assigner (Genomics England,..."
4,patient_date_of_birth,Date Of Birth,OPTIONAL,Date,PID-7.1,Patient.birthDate
5,ordering_entity_id,Original Ordering Facility Code,OPTIONAL,Code (ODS Code),ORC-21,Specimen.identifier (as received) assigner
6,glh_laboratory_id,Filler Order Ordering Facility Code,MUST,Code (ODS Code),ORC-21,ServiceRequest.requester / requisition assigne...
7,primary_sample_received_date,Sample Received Date,OPTIONAL,Date,SPM-18 (primary SPM),Specimen.receivedTime (primary specimen)
8,primary_sample_id_as_received_by_glh,Received Sample Identifier,OPTIONAL,String,SPM-2.1 (primary SPM),Specimen.identifier (as received)
9,primary_sample_id_in_glh_lims,LIMS Sample Identifier,OPTIONAL,String,SPM-2.2 (primary SPM),Specimen.identifier (GLH LIMS)


## Sample manifest data

`Input/dWGS.csv`, checked against the mapping table above: every `MUST`/`MUST (cancer
only)` field is populated, and the type-sensitive fields (`Number`, `Date`,
`String (pattern)`) parse cleanly.

In [2]:
dwgs_df = pd.read_csv("Input/dWGS.csv", dtype=str, keep_default_na=False)
print(f"{len(dwgs_df)} rows, {len(dwgs_df.columns)} columns")

assert list(dwgs_df.columns) == field_mapping_df.csv_field.tolist(), \
    "Input/dWGS.csv column order must match the field mapping table above"

# Unconditional MUST fields must not be blank. "MUST (cancer only)" fields are
# conditional - this test data is all rare-disease dWGS referrals, not cancer,
# so those are correctly left blank rather than checked here.
must_fields = field_mapping_df[field_mapping_df.cardinality == "MUST"].csv_field
for field in must_fields:
    blank_rows = dwgs_df.index[dwgs_df[field].str.strip() == ""].tolist()
    assert not blank_rows, f"{field} is MUST but blank in row(s) {blank_rows}"

# Number/Date fields must parse as their stated type.
for field in field_mapping_df[field_mapping_df.field_type == "Number"].csv_field:
    dwgs_df[field].map(lambda v: float(v) if v.strip() else None)
for field in field_mapping_df[field_mapping_df.field_type == "Date"].csv_field:
    pd.to_datetime(dwgs_df[field].replace("", pd.NaT))

# gmc_rack_well must match the SOP's stated pattern.
rack_well_pattern = r"^[A-H][0][1-9]|[A-H][1][0-2]$"
assert dwgs_df.gmc_rack_well.str.match(rack_well_pattern).all(), "gmc_rack_well values must match the SOP pattern"

print("Input/dWGS.csv satisfies every cardinality/field_type constraint above.")
dwgs_df

6 rows, 42 columns
Input/dWGS.csv satisfies every cardinality/field_type constraint above.


,referral_id,clinical_indication_test_type_id,patient_nhs_number,patient_ngis_id,patient_date_of_birth,ordering_entity_id,glh_laboratory_id,primary_sample_received_date,primary_sample_id_as_received_by_glh,primary_sample_id_in_glh_lims,...,gmc_rack_well,dna_extraction_protocol,prolonged_sample_storage,retrospective_sample,approved_by,patient_forename,patient_surname,family_structure,participant_type,clinical_information
0,r2026000201,R14.1,9737383222,p2026000101,1978-01-17,RR8,699C0,2026-08-20,RR8-P0001,YNE26-P0001,...,A01,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Rob,Leeds,Singleton,Proband,Suspected inherited neurodevelopmental condition.
1,r2026000202,R59.1,9737873971,p2026000102,1947-04-27,RTR,699N0,2026-08-20,RTR-P0001,YNE26-P0002,...,A02,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Allanys,Middlesborough,Duo,Proband,
2,r2026000202,R59.1,9999999603,p2026000103,1984-11-06,RTR,699N0,2026-08-20,RTR-P0002,YNE26-P0003,...,A03,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Fourteen,Editestpatient,Duo,Family Member,
3,r2026000203,R27.3,9737383362,p2026000104,1989-07-01,RNN,699D0,2026-08-20,RNN-P0001,YNE26-P0004,...,A04,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Gilly,Brough,Trio,Proband,Patient has intellectual disability
4,r2026000203,R27.3,9999999581,p2026000105,1960-01-01,RNN,699D0,2026-08-20,RNN-P0002,YNE26-P0005,...,A05,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Thirteen,Editestpatient,Trio,Family Member,
5,r2026000203,R27.3,9999999514,p2026000106,1988-01-14,RNN,699D0,2026-08-20,RNN-P0003,YNE26-P0006,...,A06,Qiagen,Frozen,New,Duty Scientist on behalf of NEY GMS,Six,Editestpatient,Trio,Family Member,


## Background

`Input/dWGS.csv` is a convenient stand-in, not a claim about the real world: in practice
each GLH/LIMS combination is likely to export a different local format, and this
notebook's manifest is only one plausible shape used here to get test data flowing
end-to-end. The field mapping above is the part expected to generalise; the CSV's exact
column names are not.

### Where this sits: a sub-contracted order

At a high level, North East and Yorkshire (NE&Y) Genomics referring a sample on to North
West Genomics for sequencing is a **sub-contracted order**. In NW-GMSA's own
[Inter-Laboratory Workflow (ILW) page](https://nw-gmsa.github.io/en/ILW.html), this is
**`LAB-35`** (Sub-order Management) — a laboratory (NE&Y Genomics, acting as an *Order
Placer* toward NW Genomics) sending a sub-order to another laboratory (NW Genomics,
acting as *Order Filler*) for testing it can't do itself, distinct from `LAB-1` (the
original clinical order a referring Trust placed with NE&Y Genomics in the first place).
Results come back the same way, as `LAB-36`.

Per [NW-GMSA's Message Exchange page](https://nw-gmsa.github.io/en/MQ.html), a `LAB-35`
order like this can be sent as **either** a FHIR `Bundle` (`POST [base]/$process-message`,
the [laboratory-order `MessageDefinition`](MessageDefinition-laboratory-order.html)) or
HL7 v2 `OML^O21` (MLLP, [documented here](hl7v2.html#oml_o21-laboratory-order)) — the
same choice `03-laboratory-order-from-csv.ipynb` exercised for a different kind of
order. Both follow the *same* underlying data model (next section), so the choice is
purely about what the sending system can produce.

### Why the format actually sent will vary

Which format NE&Y Genomics uses will depend on how each constituent lab implements it.
"NE&Y Genomics" isn't one building — the
[Yorkshire and North East GLH](https://directory.spineservices.nhs.uk/ORD/2-0-0/organisations/699N0)
is a partnership of laboratories, three of which `Input/dWGS.csv` mixes between as
`glh_laboratory_id` below (real ODS codes, confirmed against the live
[NHS ODS API](https://directory.spineservices.nhs.uk/ORD/2-0-0/organisations)):

- **Northern Genetics Service**, Newcastle — `699N0` (the GLH's own registration,
  based at Newcastle's Institute for Human Genetics - not `RTD07`, that site's separate
  NHS Trust Site identity, which this manifest field doesn't use)
- **Sheffield Diagnostic Genetics Service** — `699D0`
- **The Leeds Genetics Laboratory** — `699C0`

Some of these may already run a LIMS capable of sending `ORM^O01`/`OML^O21` (or FHIR)
directly; others may only export a flat file, needing a data pipeline in between to
produce a proper order message. `Input/dWGS.csv` is standing in for that second case.

![notebook 8 diagram 3](https://mermaid.ink/svg/Zmxvd2NoYXJ0IFRCCiAgICBzdWJncmFwaCBHTEhbIk5FJlkgR2Vub21pY3MiXQogICAgICAgIE5DTFsiTm9ydGhlcm4gR2VuZXRpY3MgU2VydmljZTxici8+TmV3Y2FzdGxlIC0gNjk5TjAiXQogICAgICAgIFNIRlsiU2hlZmZpZWxkIERpYWdub3N0aWM8YnIvPkdlbmV0aWNzIFNlcnZpY2UgLSA2OTlEMCJdCiAgICAgICAgTERTWyJUaGUgTGVlZHMgR2VuZXRpY3M8YnIvPkxhYm9yYXRvcnkgLSA2OTlDMCJdCiAgICBlbmQKICAgIFBJUEVbIk1hbmlmZXN0IENTViAtPiBtZXNzYWdlPGJyLz5kYXRhIHBpcGVsaW5lPGJyLz4odGhpcyBub3RlYm9vaydzIHdvcmtlZCBleGFtcGxlKSJdCiAgICBSSUVbIk5XIEdlbm9taWNzPGJyLz5SZWdpb25hbCBJbnRlZ3JhdGlvbiBFbmdpbmUgKFJJRSk8YnIvPjY5OVgwIl0KCiAgICBOQ0wgLS0gIkxJTVMgc2VuZHMgT01MXk8yMSAvIE8yMSBGSElSIE1lc3NhZ2UgZGlyZWN0bHkiIC0tPiBSSUUKICAgIExEUyAtLSAiTElNUyBzZW5kcyBPTUxeTzIxIC8gTzIxIEZISVIgTWVzc2FnZSBkaXJlY3RseSIgLS0+IFJJRQogICAgU0hGIC0tICJMSU1TIGV4cG9ydHMgYSBtYW5pZmVzdCBDU1YiIC0tPiBQSVBFCiAgICBQSVBFIC0tICJPTUxeTzIxIC8gTzIxIEZISVIgTWVzc2FnZSIgLS0+IFJJRQ==)

<!--
```mermaid
flowchart TB
    subgraph GLH["NE&Y Genomics"]
        NCL["Northern Genetics Service<br/>Newcastle - 699N0"]
        SHF["Sheffield Diagnostic<br/>Genetics Service - 699D0"]
        LDS["The Leeds Genetics<br/>Laboratory - 699C0"]
    end
    PIPE["Manifest CSV -> message<br/>data pipeline<br/>(this notebook's worked example)"]
    RIE["NW Genomics<br/>Regional Integration Engine (RIE)<br/>699X0"]

    NCL -- "LIMS sends OML^O21 / O21 FHIR Message directly" --> RIE
    LDS -- "LIMS sends OML^O21 / O21 FHIR Message directly" --> RIE
    SHF -- "LIMS exports a manifest CSV" --> PIPE
    PIPE -- "OML^O21 / O21 FHIR Message" --> RIE
```
-->

### The shared data model

Both the v2 and FHIR `O21` orders follow the same model, defined once in the NW
Implementation Guide and rendered as whichever wire format the sender chooses:

![notebook 8 diagram 4](https://mermaid.ink/svg/ZXJEaWFncmFtCiAgICBNZXNzYWdlSGVhZGVyIHx8LS18fCBTZXJ2aWNlUmVxdWVzdCA6IGZvY3VzCiAgICBQYXRpZW50IHx8LS1veyBTZXJ2aWNlUmVxdWVzdCA6IHN1YmplY3QKICAgIFNlcnZpY2VSZXF1ZXN0IHx8LS1veyBTcGVjaW1lbiA6IHNwZWNpbWVuCiAgICBTZXJ2aWNlUmVxdWVzdCB8fC0tb3sgT2JzZXJ2YXRpb24gOiBzdXBwb3J0aW5nSW5mbw==)

<!--
```mermaid
erDiagram
    MessageHeader ||--|| ServiceRequest : focus
    Patient ||--o{ ServiceRequest : subject
    ServiceRequest ||--o{ Specimen : specimen
    ServiceRequest ||--o{ Observation : supportingInfo
```
-->

- **`MessageHeader`** (`MSH`)
- **`Patient`** (`PID`)
- **`ServiceRequest`** (`ORC` + `OBR`)
- **`Specimen`** (`SPM`)
- **`Observation`** (`OBX`) — *not* clinical results here, but "ask at order entry"
  supplemental questions and answers (e.g. `family_structure`, `participant_type`),
  referenced from `ServiceRequest.supportingInfo` — the mapping this notebook's earlier
  `FIELD_MAPPING` table originally guessed at with a `(proposed)` extension, now
  corrected above to use the home NW-GMSA's own order model already defines for exactly
  this case

**NW Genomics' own LIMS speaks HL7 v2 internally, always** — regardless of which format
this sub-order arrives in, the RIE is what transforms it (FHIR or v2) into NW Genomics'
internal v2 shape before it ever reaches the LIMS. The worked example below builds the
external-facing FHIR order and converts it to v2 the same way the RIE's own
`transformToV2` tooling does — it does not reach into the LIMS's internal format.

### Room to extend beyond what's specified

Although the NW data model above names specific FHIR and v2 elements, it doesn't forbid
others - both standards still accept elements it doesn't mention, as long as they're
individually conformant to the schema (and, on the FHIR side, pass validation - this
repo's own RIE converts inbound v2 to FHIR and validates that internally too, the same
check `03-laboratory-order-from-csv.ipynb`/`04-laboratory-report-fhir-from-hl7v2.ipynb`
run explicitly). So NE&Y Genomics is free to add fields of its own - exactly what the
5 NEY local-extension fields in the field-mapping table above already are - as long as
they stay conformant to both schemas.

This is the main advantage either HL7 standard has over exchanging flat files like
`Input/dWGS.csv` directly: producer and consumer aren't tightly coupled to one exact,
frozen column layout the way a raw CSV exchange would be. A new field can be added by
one side without breaking the other, which is what actually makes a CSV-based interface
brittle and costly to change compared to a schema built to expect extension.

(A worked example of the reverse direction - HL7 v2 into FHIR - is
`04-laboratory-report-fhir-from-hl7v2.ipynb`, which converts a `ORU^R01` laboratory
report the same hand-built way this notebook builds an order.)

## FHIR worked example

Same approach as `03-laboratory-order-from-csv.ipynb`: build the order by hand in
Python, validate resource-by-resource, then assemble and validate the whole `Bundle`.
Row 0 of `Input/dWGS.csv` (referral `r2026000201`, Rob Leeds, NHS number `9737383222` -
the same patient used throughout `03`-`05`) is the source data.

To keep this tractable, the worked example only builds the fields that already have a
confirmed (non-blank) `hl7v2_field`/`fhir_field` in the mapping table above. Every
`OPTIONAL` field still mapped as `""`/`""` (topography, morphology, QC metrics, rack
position, and so on) is a genuine open question for a future pass, not an oversight -
see the closing note.

In [3]:
import json
import os
import subprocess
import tempfile
from datetime import datetime
from uuid import uuid4

import requests
from dotenv import load_dotenv

load_dotenv()

toolsServer = os.getenv("V2_TOOLS")

NHS_NUMBER_SYSTEM = "https://fhir.nhs.uk/Id/nhs-number"
ODS_SYSTEM = "https://fhir.nhs.uk/Id/ods-organization-code"
V2_0203 = "http://terminology.hl7.org/CodeSystem/v2-0203"
GENOMIC_TEST_DIRECTORY_SYSTEM = "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory"
NW_GLH_ODS = "699X0"        # NW Genomics - the LAB-35 sub-order's Order Filler
GENOMICS_ENGLAND_ODS = "8J834"  # Genomics England - runs NGIS (National Genomic Informatics
                                 # System), the assigner of patient_ngis_id below

# The three NE&Y Genomics constituent labs dWGS.csv's glh_laboratory_id mixes between -
# real ODS codes, confirmed against the live NHS ODS API (see Background above).
NEY_GLH_NAME = {
    "699N0": "Northern Genetics Service, Newcastle (NE&Y Genomics)",
    "699D0": "Sheffield Diagnostic Genetics Service (NE&Y Genomics)",
    "699C0": "The Leeds Genetics Laboratory (NE&Y Genomics)",
}

NWGMSA_PATIENT_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/Patient"
NWGMSA_SERVICE_REQUEST_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/ServiceRequest"
NWGMSA_SPECIMEN_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/Specimen"


def details(item):
    return item["text"]


def issues_df(outcome):
    df = pd.DataFrame(outcome["issue"])
    df["details"] = df["details"].apply(details)
    df.drop(columns=["extension"], inplace=True, errors="ignore")
    df.sort_values(by=["severity"], inplace=True)
    df = df[~df["details"].str.contains("ValueSet/mimetypes")]
    df = df[~df["details"].str.contains("failed: dom-6")]
    df = df[~df["details"].str.contains("bcp:13")]
    df = df[~df["details"].str.contains("no terminology service")]
    return df


def validate_resource(resource, profile):
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp:
        json.dump(resource, tmp)
        tmp_path = tmp.name
    outcome_path = tmp_path + "-OperationOutcome.json"
    subprocess.run(
        [
            "java", "-jar", "validator_cli.jar", tmp_path,
            "-version", "4.0.1", "-ig", "package.tgz",
            "-profile", profile, "-tx", "n/a",
            "-output", outcome_path, "-output-style", "json",
        ],
        capture_output=True,
    )
    with open(outcome_path) as f:
        return issues_df(json.load(f))


row = dwgs_df.iloc[0]
row[["referral_id", "clinical_indication_test_type_id", "patient_nhs_number", "patient_ngis_id",
     "ordering_entity_id", "glh_laboratory_id", "primary_sample_id_as_received_by_glh",
     "dispatched_sample_lsid", "glh_sample_consignment_number", "family_structure", "participant_type"]]

referral_id                                      r2026000201
clinical_indication_test_type_id                       R14.1
patient_nhs_number                                9737383222
patient_ngis_id                                  p2026000101
ordering_entity_id                                       RR8
glh_laboratory_id                                      699C0
primary_sample_id_as_received_by_glh               RR8-P0001
dispatched_sample_lsid                           FX000000001
glh_sample_consignment_number           NEY-CONSIGN-2026-100
family_structure                                   Singleton
participant_type                                     Proband
Name: 0, dtype: object

### Patient

`patient_ngis_id` has no confirmed FHIR identifier *system* URI, but it does have a
confirmed **assigner**: NGIS (the National Genomic Informatics System) is a service run
by Genomics England, whose ODS code is `8J834` (confirmed against the live
[NHS ODS API](https://directory.spineservices.nhs.uk/ORD/2-0-0/organisations/8J834)) -
so rather than guess at a `system` URI, the identifier below carries `assigner` instead,
the same ODS-organisation-code pattern `Specimen.identifier.assigner` uses further down.
This resolves the field-mapping table's earlier `(TBC system)` flag for this field.
`type`, meanwhile, has an honest answer even without a confirmed `system`: v2-0203's
generic `PI` ("Patient internal identifier") fits exactly what this is - some other
system's internal ID for this patient - so it's coded rather than left text-only.

In [4]:
patient_fullurl = f"urn:uuid:{uuid4()}"
patient = {
    "resourceType": "Patient",
    "identifier": [
        {"system": NHS_NUMBER_SYSTEM, "type": {"coding": [{"system": V2_0203, "code": "NH"}]}, "value": row["patient_nhs_number"]},
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": GENOMICS_ENGLAND_ODS}},
         "type": {"coding": [{"system": V2_0203, "code": "PI"}]}, "value": row["patient_ngis_id"]},
    ],
    "name": [{"family": row["patient_surname"], "given": [row["patient_forename"]]}],
    "birthDate": row["patient_date_of_birth"],
}
print(json.dumps(patient, indent=2))

{
  "resourceType": "Patient",
  "identifier": [
    {
      "system": "https://fhir.nhs.uk/Id/nhs-number",
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "NH"
          }
        ]
      },
      "value": "9737383222"
    },
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "8J834"
        }
      },
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "PI"
          }
        ]
      },
      "value": "p2026000101"
    }
  ],
  "name": [
    {
      "family": "Leeds",
      "given": [
        "Rob"
      ]
    }
  ],
  "birthDate": "1978-01-17"
}


In [5]:
validate_resource(patient, NWGMSA_PATIENT_PROFILE)

,severity,code,details,expression


### Two Observations for the "ask at order" questions

`family_structure` and `participant_type` are enumerated-string answers to questions
asked at the point of ordering, not clinical findings - so `code` is text-only (no
NW-GMSA-confirmed coding system exists for either, same reasoning the field-mapping
table already gives). There's no dedicated NW-GMSA profile for this "ask at order"
pattern to validate against individually - a real gap this worked example surfaces
rather than works around.

In [6]:
def ask_at_order_observation(question_text, answer_text, patient_ref):
    return {
        "resourceType": "Observation",
        "status": "final",
        "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "exam"}]}],
        "code": {"text": question_text},
        "subject": {"reference": patient_ref},
        "valueCodeableConcept": {"text": answer_text},
    }


family_structure_fullurl = f"urn:uuid:{uuid4()}"
family_structure_observation = ask_at_order_observation("Family Structure", row["family_structure"], patient_fullurl)

participant_type_fullurl = f"urn:uuid:{uuid4()}"
participant_type_observation = ask_at_order_observation("Participant Type", row["participant_type"], patient_fullurl)

print(json.dumps(family_structure_observation, indent=2))

{
  "resourceType": "Observation",
  "status": "final",
  "category": [
    {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/observation-category",
          "code": "exam"
        }
      ]
    }
  ],
  "code": {
    "text": "Family Structure"
  },
  "subject": {
    "reference": "urn:uuid:39ad51b6-f141-44c9-aa05-807889b3993d"
  },
  "valueCodeableConcept": {
    "text": "Singleton"
  }
}


### Specimen

One `Specimen`, not two: `dWGS.csv`'s primary-specimen and dispatched-specimen fields
both describe the same physical sample as it moves through this sub-order, so they're
carried as multiple identifiers on a single resource rather than two linked resources.
Two of those identifiers have different **assigners**, reflecting who actually issued
each one:

- **Received Sample Identifier** (`primary_sample_id_as_received_by_glh`) - assigned by
  `ordering_entity_id` (Leeds, `RR8`), the *original* order placer who received and
  labelled the specimen before it was sent on. Coded `type` `PLAC` ("Placer
  Identifier") - the identifier assigned by whoever placed the order/collected the
  specimen.
- **LIMS Sample Identifier** (`primary_sample_id_in_glh_lims`) - assigned by
  `glh_laboratory_id` (`699N0`), NE&Y Genomics' own LIMS, once the specimen was logged
  there. Coded `type` `FILL` ("Filler Identifier") - the identifier assigned by the
  party filling/processing the order.

`glh_sample_consignment_number` becomes the `identifier:ShipmentTrackingNumber` slice,
coded `type` `STN` ("Shipment Tracking Number"), per the field-mapping table above.
All three `type` codes are from
[HL7's v2-0203 CodeSystem](http://terminology.hl7.org/CodeSystem/v2-0203) - the same
system already used for `Patient.identifier.type` above - rather than left text-only.
`primary_sample_type`/`primary_sample_state`/`dispatched_sample_state` do stay
text-only, though - this CSV gives free text (`"blood_unsorted_edta"`, `"DNA"`), not the
SNOMED CT codes the profile's `type` binding actually wants, and the field-mapping table
already flags `primary_sample_type` as low-confidence.

In [7]:
specimen_fullurl = f"urn:uuid:{uuid4()}"
specimen = {
    "resourceType": "Specimen",
    "identifier": [
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": row["ordering_entity_id"]}},
         "type": {"coding": [{"system": V2_0203, "code": "PLAC"}]}, "value": row["primary_sample_id_as_received_by_glh"]},
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": row["glh_laboratory_id"]}},
         "type": {"coding": [{"system": V2_0203, "code": "FILL"}]}, "value": row["primary_sample_id_in_glh_lims"]},
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": row["glh_laboratory_id"]}},
         "type": {"coding": [{"system": V2_0203, "code": "STN"}]}, "value": row["glh_sample_consignment_number"]},
    ],
    "type": {"text": row["primary_sample_state"]},
    "subject": {"reference": patient_fullurl},
    "collection": {"collectedDateTime": row["received_sample_collection_date"] + "T00:00:00+00:00"},
    "receivedTime": row["primary_sample_received_date"] + "T00:00:00+00:00",
    "container": [{"identifier": [{"value": row["dispatched_sample_lsid"]}]}],
}

print(json.dumps(specimen, indent=2))

{
  "resourceType": "Specimen",
  "identifier": [
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "RR8"
        }
      },
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "PLAC"
          }
        ]
      },
      "value": "RR8-P0001"
    },
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "699C0"
        }
      },
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "FILL"
          }
        ]
      },
      "value": "YNE26-P0001"
    },
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "699C0"
        }
      },
      "type": {
        "coding": [


In [8]:
validate_resource(specimen, NWGMSA_SPECIMEN_PROFILE)

,severity,code,details,expression
1,error,code-invalid,"No code provided, and a code is required from ...",[Specimen.type]
0,warning,code-invalid,None of the codings provided are in the value ...,[Specimen.identifier[2].type]


The `error`-severity `Specimen.type` row above isn't cosmetic - `dWGS.csv`
genuinely only has free text (`"blood_unsorted_edta"`, `"DNA"`) for a field the profile
requires a SNOMED CT code for, and no lookup table from these enum values to SNOMED
exists yet anywhere in this repo. It doesn't stop the HL7 v2 transform below from
running, but the "Checking the transform" section shows it does silently disappear from
the v2 output rather than surviving as free text - arguably a worse outcome than a loud
failure.

### ServiceRequest

For this `LAB-35` sub-order, `requester` is `glh_laboratory_id` (whichever constituent
NE&Y Genomics lab this row's sample went through) - the party actually placing *this*
order with NW Genomics - not `ordering_entity_id` (the *original* referring Trust).
`requisition` (`referral_id`) is likewise assigned by `glh_laboratory_id`, since it's
NE&Y Genomics' own group number for the sub-order, not the referring Trust's. The
referring Trust's role is preserved instead as the `Specimen` identifier assigner above
- the original order placer's involvement doesn't disappear, it just isn't
`ServiceRequest.requester` for *this* message.

`intent` is `filler-order`, not the plain `order` `03-laboratory-order-from-csv.ipynb`
used - this `ServiceRequest` is NW Genomics (the filler) restating NE&Y Genomics'
sub-order, not an original order in its own right.

NW-GMSA's profile also requires at least one `identifier` (separate from `requisition`,
which is the *group* number). `dWGS.csv` has no distinct placer-order-number column, so
- as `03-laboratory-order-from-csv.ipynb` did for a similarly-missing field - `referral_id`
is reused as a stand-in `PLAC` identifier too, rather than left out.

In [9]:
service_request_fullurl = f"urn:uuid:{uuid4()}"
service_request = {
    "resourceType": "ServiceRequest",
    "status": "active",
    "intent": "filler-order",
    "category": [{"coding": [{"system": "http://snomed.info/sct", "code": "116148004"}]}],
    "code": {"coding": [{"system": GENOMIC_TEST_DIRECTORY_SYSTEM, "code": row["clinical_indication_test_type_id"]}]},
    "requisition": {
        "assigner": {"identifier": {"system": ODS_SYSTEM, "value": row["glh_laboratory_id"]}},
        "value": row["referral_id"],
    },
    "identifier": [
        {"assigner": {"identifier": {"system": ODS_SYSTEM, "value": row["glh_laboratory_id"]}},
         "type": {"coding": [{"system": V2_0203, "code": "PLAC"}]}, "value": row["referral_id"]},  # stand-in - see above
    ],
    "subject": {"reference": patient_fullurl},
    "requester": {
        "display": NEY_GLH_NAME[row["glh_laboratory_id"]],
        "identifier": {"system": ODS_SYSTEM, "value": row["glh_laboratory_id"]},
        "type": "Organization",
    },
    "specimen": [{"reference": specimen_fullurl, "type": "Specimen"}],
    "supportingInfo": [
        {"reference": family_structure_fullurl, "type": "Observation"},
        {"reference": participant_type_fullurl, "type": "Observation"},
    ],
    "note": [{"text": row["clinical_information"]}],
}

print(json.dumps(service_request, indent=2))

{
  "resourceType": "ServiceRequest",
  "status": "active",
  "intent": "filler-order",
  "category": [
    {
      "coding": [
        {
          "system": "http://snomed.info/sct",
          "code": "116148004"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory",
        "code": "R14.1"
      }
    ]
  },
  "requisition": {
    "assigner": {
      "identifier": {
        "system": "https://fhir.nhs.uk/Id/ods-organization-code",
        "value": "699C0"
      }
    },
    "value": "r2026000201"
  },
  "identifier": [
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "699C0"
        }
      },
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "PLAC"
          }
        ]
      },
      "value": "r2026000201"
    }
  ],
 

In [10]:
validate_resource(service_request, NWGMSA_SERVICE_REQUEST_PROFILE)

,severity,code,details,expression
2,error,structure,ServiceRequest.authoredOn: minimum required = ...,[ServiceRequest]
0,information,informational,This element does not match any known slice de...,[ServiceRequest.supportingInfo[0]]
1,information,informational,This element does not match any known slice de...,[ServiceRequest.supportingInfo[1]]
4,information,code-invalid,None of the codings provided are in the value ...,[ServiceRequest.code]
3,warning,not-found,A definition for the value Set 'http://hl7.org...,[ServiceRequest.code]


The remaining `error` (`ServiceRequest.authoredOn: minimum required = 1`) is the
same pre-existing NW-GMSA profile gap `04-laboratory-report-fhir-from-hl7v2.ipynb`
already found and confirmed against this repo's real production output: nothing in
`dWGS.csv` records when the order was authored, so - as there - it's left out rather
than filled with an invented date.

### MessageHeader and the Bundle

`sender` is NE&Y Genomics (`699N0`, this `LAB-35` sub-order's Order Placer);
`destination` is NW Genomics (`699X0`, the Order Filler). `Bundle.identifier`/
`.timestamp` are mandatory on the
[`Bundle` (message) profile](https://nw-gmsa.github.io/en/StructureDefinition-BundleMessage.html),
as throughout this series.

In [11]:
message_header = {
    "resourceType": "MessageHeader",
    "eventCoding": {"system": "http://terminology.hl7.org/CodeSystem/v2-0003", "code": "O21"},
    "sender": {"identifier": {"system": ODS_SYSTEM, "value": row["glh_laboratory_id"]}},
    "destination": [{"endpoint": "https://fhir.nwgenomics.nhs.uk/Endpoint/RIE",
                      "receiver": {"identifier": {"system": ODS_SYSTEM, "value": NW_GLH_ODS}}}],
    "source": {"endpoint": "https://fhir.nwgenomics.nhs.uk/Endpoint/NEYGenomics", "software": "NE&Y Genomics"},
    "focus": [{"reference": service_request_fullurl}],
}

order_bundle = {
    "resourceType": "Bundle",
    "identifier": {"value": f"urn:uuid:{uuid4()}"},
    "timestamp": datetime.now().astimezone().strftime("%Y-%m-%dT%H:%M:%S+00:00"),
    "type": "message",
    "entry": [
        {"fullUrl": f"urn:uuid:{uuid4()}", "resource": message_header},
        {"fullUrl": patient_fullurl, "resource": patient},
        {"fullUrl": specimen_fullurl, "resource": specimen},
        {"fullUrl": family_structure_fullurl, "resource": family_structure_observation},
        {"fullUrl": participant_type_fullurl, "resource": participant_type_observation},
        {"fullUrl": service_request_fullurl, "resource": service_request},
    ],
}

order_filename = f"dWGS_{row['referral_id']}.json"
with open("Input/FHIR/O21/" + order_filename, "w") as f:
    json.dump(order_bundle, f, indent=2)

print("Saved Input/FHIR/O21/" + order_filename)

Saved Input/FHIR/O21/dWGS_r2026000201.json


In [12]:
outcome_path = "Results/FHIR/O21/" + order_filename + "-OperationOutcome.json"
subprocess.run(
    [
        "java", "-jar", "validator_cli.jar", "Input/FHIR/O21/" + order_filename,
        "-version", "4.0.1", "-ig", "package.tgz",
        "-bundle", "ServiceRequest:0", NWGMSA_SERVICE_REQUEST_PROFILE, "-tx", "n/a",
        "-output", outcome_path, "-output-style", "json",
    ],
    capture_output=True,
)
with open(outcome_path) as f:
    issues_df(json.load(f))

## HL7 v2 worked example

Same `transformToV2` tooling endpoint used throughout this series. An earlier draft of
this `Bundle` - before the `Patient.identifier[1].type` fix above (`coding` + `text`
together, which turned out to violate a fixed value the profile pins that slice to) -
got a `500` back with an empty body from this same call. That's worth knowing even
though it's fixed now: a structural profile violation on `Patient` was enough to fail
the *whole* transform, not just that one field, with nothing in the response to say
why - the `FHIR Validation.ipynb`-style checks above are what actually located it.

In [13]:
with open("Input/FHIR/O21/" + order_filename, "rb") as f:
    order_json = f.read()

rV2 = requests.post(toolsServer + "/transformToV2", data=order_json, verify=False, headers={"Content-Type": "application/fhir+json"})

print(rV2.status_code)

v2_filename = order_filename.replace(".json", ".txt")
with open("Output/V2/O21/" + v2_filename, "w") as f:
    f.write(rV2.text)

print(rV2.text)

200




### Sending the order onward

The natural next step for a real `LAB-35` sub-order — `POST`ing `order_bundle` to
`FHIR_SERVER`'s `$process-message` endpoint, the way `Testing.ipynb`'s `O21`/`O01`
loops do for other message types — isn't exercised here: NE&Y Genomics and NW
Genomics sit in different network environments, and a direct synchronous
`$process-message` call between them hits firewall restrictions. **Note for the real
implementation, not something this notebook builds**: production delivery for this
inter-organisation route is expected to go via a message queue (AWS SQS) rather than a
direct HTTP POST across that boundary.

## Checking the transform

A `200` back doesn't mean everything survived the trip - check every field value this
notebook put into the FHIR `Bundle` for whether it actually shows up somewhere in the
resulting v2 message.

In [14]:
check_values = {
    "referral_id (requisition)": row["referral_id"],
    "clinical_indication_test_type_id (ServiceRequest.code)": row["clinical_indication_test_type_id"],
    "patient_nhs_number": row["patient_nhs_number"],
    "patient_forename": row["patient_forename"],
    "patient_surname": row["patient_surname"],
    "ordering_entity_id (requester ODS)": row["ordering_entity_id"],
    "primary_sample_id_as_received_by_glh": row["primary_sample_id_as_received_by_glh"],
    "primary_sample_id_in_glh_lims": row["primary_sample_id_in_glh_lims"],
    "primary_sample_state (Specimen.type text)": row["primary_sample_state"],
    "dispatched_sample_lsid (container.identifier)": row["dispatched_sample_lsid"],
    "dispatched_sample_state (Specimen.type text)": row["dispatched_sample_state"],
    "glh_sample_consignment_number": row["glh_sample_consignment_number"],
    "family_structure (Observation.valueCodeableConcept text)": row["family_structure"],
    "participant_type (Observation.valueCodeableConcept text)": row["participant_type"],
    "clinical_information": row["clinical_information"],
    "NE&Y Genomics sender ODS": row["glh_laboratory_id"],
    "NW Genomics destination ODS": NW_GLH_ODS,
}

pd.DataFrame(
    [{"field": field, "value": value, "present_in_v2": value in rV2.text} for field, value in check_values.items()]
)

,field,value,present_in_v2
0,referral_id (requisition),r2026000201,True
1,clinical_indication_test_type_id (ServiceReque...,R14.1,True
2,patient_nhs_number,9737383222,True
3,patient_forename,Rob,True
4,patient_surname,Leeds,True
5,ordering_entity_id (requester ODS),RR8,True
6,primary_sample_id_as_received_by_glh,RR8-P0001,True
7,primary_sample_id_in_glh_lims,YNE26-P0001,False
8,primary_sample_state (Specimen.type text),blood_unsorted_edta,False
9,dispatched_sample_lsid (container.identifier),FX000000001,False


Coding the `Specimen` identifiers' `type` (`PLAC`/`FILL`/`STN`, added above) changes
which ones survive, but doesn't make the mystery go away - it sharpens it. `PLAC`
(Received Sample Identifier) and `STN` (Shipment Tracking Number) both now come through,
each landing in its own `SPM-2` repeat/component along with its assigner (`RR8`/`699C0`)
- so `ordering_entity_id` and `primary_sample_id_as_received_by_glh` flip from `False` to
`True`. `FILL` (LIMS Sample Identifier), though, is still `False` even coded - so this
isn't simply "coded survives, text-only doesn't": the transform's `SPM-2` mapping
recognises `PLAC` and `STN` specifically and drops `FILL`, for a reason this notebook
hasn't chased down. Everything still text-only - `Specimen.type`, both "ask at order"
`Observation.valueCodeableConcept`s - still vanishes exactly as before, consistent with
the coded-survives-text-doesn't pattern holding for *those* fields even where it isn't
the whole story for identifiers. Every genuinely coded value elsewhere (NHS number, ODS
codes on `MessageHeader`/`requester`, the Genomic Test Directory code) still comes
through fine. Why `FILL` specifically doesn't map is a real, worth-chasing question this
notebook surfaces rather than answers - exactly its job at this stage.

## Look forward: what comes back

This order is one half of a closed loop - [NW-GMSA's ILW
page](https://nw-gmsa.github.io/en/ILW.html) pairs this `LAB-35` sub-order with a
`LAB-36` result coming back the other way. Which shape that result takes depends on
what the requester actually needs:

- **A PDF report** - `LAB-3` (`ORU^R01`/FHIR `R01` Message), the narrative report
  `04-laboratory-report-fhir-from-hl7v2.ipynb` builds.
- **Structured variants** - `LAB-5` (`ORU^R01`/FHIR `R01` Message), the discrete
  genomic findings `05-test-results-from-vcf.ipynb` builds.
- **Both together** - an XD-LAB/FHIR Document, combining a PDF and structured results
  in one document `Bundle`, which `06-eu-laboratory-report-fhir-document.ipynb` builds.
  This is the shape most likely to matter here: it's the format NHS England's Unified
  Genomic Record (UGR) is expected to use.

## Closing note

This is a preparation/analysis notebook, not a settled spec. Running the worked example
above already surfaced real findings this notebook's earlier analysis alone hadn't:
corrections to the field-mapping table (`family_structure`/`participant_type` moving
from a guessed extension to `Observation`/`supportingInfo`, `glh_laboratory_id` moving
off `ServiceRequest.requester` onto `MessageHeader.sender`), a structural `Patient`
mistake that failed the *entire* v2 transform with no explanation until the FHIR
validation step caught it, and - via the discrepancy check - a partially-explained
pattern in which text-only values reliably vanish in that same transform, but coded
`Specimen` identifiers can too depending on which v2-0203 type code they carry (`PLAC`/
`STN` survive, `FILL` doesn't) - not something coding the identifiers was expected to
leave open. Exactly the kind of thing this notebook is *for*. Expect further passes to
keep finding gaps like these, which may in turn mean `Input/dWGS.csv` itself needs
updating (new columns, corrected values) - which would then change this worked example
again. That loop is the intended way this notebook gets used, not a sign something went
wrong.